In [1]:
import requests, feedparser, json,time, os
import polars as pl
from config_info import APIS
from pprint import pprint
from lxml import etree

# Basic Fetch

In [2]:
def fetch_raw(url, headers=None):
    headers = headers or {"User-Agent":  "IntelliCorpus/1.0 (contact: paull@scholar-cergy.com)"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    content_type = r.headers.get("Content-Type","")
    if 'xml' in content_type:
        return r.text
    elif 'json' in content_type:
        return r.json()
    else:
        return r.text

# Arxiv 

In [3]:
def parser_arxiv(raw_result : str):
    feed = feedparser.parse(raw_result)
    results = []
    for e in feed.entries:
        results.append({
            "id": e.get("id"),
            "title": e.get("title"),
            "summary": e.get("summary"),
            "published_at": e.get("published"),
            "updated": e.get("updated"),
            "authors": [a.name for a in e.get("authors", [])],
            "pdf_url": next((link.href for link in e.links if link.type == "application/pdf"), None),
            "source": "arxiv"
        })
    return results

# Hal

In [4]:
def get_pdf_hal(link : str):
    return 0

In [5]:
def parser_hal(feed : str):
    if isinstance(feed, str):
        feed = json.loads(feed)
    docs = feed.get("response", {}).get("docs", [])
    results = []
    for doc in docs:
        results.append({
            "id": doc.get("docid"),
            "title" : doc.get("title_s"),
            "summary" : doc.get("abstract_s"),
            "published" : doc.get("publicationDate_s"),
            "authors" : [a for a in doc.get("authFullName_s",[])],
            "uri": doc.get("uri_s"),
            "pdf_url" : doc.get("files_s"),
            "source": "hal"
        })
    return results

# Pubmed        

In [6]:

from typing import List, Dict
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
DB = "pubmed"


def pubmed_search(
    query: str,
    retmax: int = 100,
    retstart: int = 0,
    email: str | None = None,
    api_key: str | None = None) -> Dict:
    """
    Step 1: Search PubMed and store results on NCBI server (usehistory)
    """
    url = f"{EUTILS_BASE}/esearch.fcgi"

    params = {
        "db": DB,
        "term": query,
        "retmax": retmax,
        "retstart": retstart,
        "usehistory": "y",
        "retmode": "json",
    }

    if email:
        params["email"] = email
    if api_key:
        params["api_key"] = api_key

    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()
    return response.json()["esearchresult"]


def pubmed_fetch(
    webenv: str,
    query_key: str,
    batch_size: int = 100,
    email: str | None = None,
    api_key: str | None = None
) -> str:
    """
    Step 2: Fetch PubMed records using WebEnv + QueryKey (XML)
    """
    url = f"{EUTILS_BASE}/efetch.fcgi"

    params = {
        "db": DB,
        "query_key": query_key,
        "WebEnv": webenv,
        "retmax": batch_size,
        "retmode": "xml",
    }

    if email:
        params["email"] = email
    if api_key:
        params["api_key"] = api_key

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.text


def fetch_pubmed(query: str, max_results: int = 200, batch_size: int = 100, delay: float = 0.34, email: str | None = None, api_key: str | None = None) -> List[str]:
    """
    High-level generator:
    - search
    - iterate over result pages
    - fetch XML batches
    """
    search_result = pubmed_search(
        query=query,
        retmax=max_results,
        email=email,
        api_key=api_key
    )
    count = int(search_result["count"])
    webenv = search_result["webenv"]
    query_key = search_result["querykey"]

    results_xml = []

    for start in range(0, min(count, max_results), batch_size):
        xml = pubmed_fetch(
            webenv=webenv,
            query_key=query_key,
            batch_size=batch_size,
            email=email,
            api_key=api_key,
        )
        results_xml.append(xml)
        time.sleep(delay)  # respect NCBI rate-limit
        # pprint(xml)
    return results_xml


# Semantic Scholar

In [7]:
def semantic_fetch(results):
    for result in results["data"]:
        print(result)
        # query = APIS["Semantic Scholar"]["paper_url"].format(paper_id=result["paperId"])
        # res = fetch_raw(query)
        # print(res)
        # print(query)

# CORE

In [8]:
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.getenv("CORE_API_KEY")
# url = "https://api.core.ac.uk/v3/search/works"

# headers = {
#     "Authorization": f"Bearer {API_KEY}"
# }

# params = { 
#     "q": "AI agent",
#     "limit": 5
# }

# r = requests.get(url, headers=headers, params=params)
# r.raise_for_status()

# data = r.json()
# # print(data.keys())
# pprint(data)


In [9]:
pprint(data["results"])
for r in data["results"]:
    print(r)

NameError: name 'data' is not defined

# Main 

In [10]:
raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
result_arxiv =  parser_arxiv(raw_result)
print(type(result_arxiv)) # Arxiv Ok 
# Test HAL
# url_hal = fetch_raw(APIS["HAL"]["api_url"].format(query="AI agent",quantity='2'))
# result_hal = parser_hal(url_hal)
# pprint(result_hal) Good 

# Test PubMed
# xml_batches = fetch_pubmed(
#     query="AI agent",
#     max_results=50,
#     batch_size=20,
#     email="proliquer@scholar-perigueuxu.com"
# )
# pprint(xml_batches)

# Test Sementic Scholar
# url_sem = fetch_raw(APIS["Semantic Scholar"]["api_url"].format(query="AI agent",quantity='5'))
# print(type(url_sem))
# semantic_fetch(url_sem)

<class 'list'>


In [11]:
for res in result_arxiv:
    pprint(res)

import polars as pl
def normalize_data(raw_data: list[dict], source_name: str, column_mapping: dict):
    if not raw_data:
        return []
    
    df = pl.DataFrame(raw_data)
    columns_to_rename = {k: v for k, v in column_mapping.items() if k in df.columns}
    df = df.rename(columns_to_rename)
    
    df = df.with_columns(
        pl.lit(source_name).alias("source")
    )
    
    df = df.unique(subset=["id"])
    
    return df.to_dicts()

{'authors': ['Qingyao Ai', 'Jingtao Zhan', 'Yiqun Liu'],
 'id': 'http://arxiv.org/abs/2501.02842v1',
 'pdf_url': 'https://arxiv.org/pdf/2501.02842v1',
 'published_at': '2025-01-06T08:38:29Z',
 'source': 'arxiv',
 'summary': 'The chapter discusses the foundational impact of modern '
            'generative AI models on information access (IA) systems. In '
            'contrast to traditional AI, the large-scale training and superior '
            'data modeling of generative AI models enable them to produce '
            'high-quality, human-like responses, which brings brand new '
            'opportunities for the development of IA paradigms. In this '
            'chapter, we identify and introduce two of them in details, i.e., '
            'information generation and information synthesis. Information '
            'generation allows AI to create tailored content addressing user '
            'needs directly, enhancing user experience with immediate, '
            'relevant output

In [12]:
from database.postgres.crud import upsert_data
from models.postgres.corpus_schema import document_table 
from config.db_engine import get_db_engine
arxiv_mapping = {
    "published_at": "published",
}

clean_arxiv_data = normalize_data(
    raw_data=result_arxiv, 
    source_name="arXiv", 
    column_mapping=arxiv_mapping
)

engine = get_db_engine()


In [13]:
upsert_data(clean_arxiv_data, ['id'], document_table, engine)

AttributeError: 'Insert' object has no attribute 'excluded'